In [109]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import re
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(63459439)
block_size = 100
batch_size = 1000
training_loops = 1000

In [87]:
with open("wizard of oz.txt", 'r', encoding='utf-8') as f:
    text = f.read()

# text = text.lower()
# text = re.findall(r'\b\w+\b', text)
int_to_char = sorted(set(text))
char_to_int = { char:i for i, char in enumerate(int_to_char) }
vocab_size = len(int_to_char)

print(vocab_size)

81


In [98]:
def encode(s):
    res = []
    for i in s:
        res.append(char_to_int[i])
    return res

def decode(arr):
    res = ""
    for i in arr:
        res += int_to_char[i]
    return res

In [90]:
data = torch.tensor(encode(text), dtype=torch.long)

split = int(len(data) * 0.8)
val_data = data[split:]
train_data = data[:split]

In [91]:
def get_batch(batch_type: str):
    data = val_data if batch_type == 'test' else train_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y

In [92]:
print(get_batch('train'))

(tensor([[73, 68,  1,  ..., 54, 71, 57],
        [27, 39, 40,  ...,  1,  8,  0],
        [78,  1, 59,  ..., 73,  1, 73],
        ...,
        [10, 54, 67,  ..., 72,  1, 68],
        [57, 68, 74,  ..., 71, 67, 58],
        [73, 61, 58,  ..., 60, 71, 54]]), tensor([[68,  1, 68,  ..., 71, 57, 58],
        [39, 40, 49,  ...,  8,  0,  0],
        [ 1, 59, 68,  ...,  1, 73, 61],
        ...,
        [54, 67, 57,  ...,  1, 68, 75],
        [68, 74, 55,  ..., 67, 58, 57],
        [61, 58, 71,  ..., 71, 54, 67]]))


In [93]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        else: 
            loss = None
        
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim = 1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [99]:
model = BigramModel(vocab_size)
xb, yb = get_batch('train')
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

#used to kick off generation
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=1000)[0].tolist()))

torch.Size([100000, 81])
tensor(4.9363, grad_fn=<NllLossBackward0>)

?;oH'mDJ.p*cWJU&C?HL:6t'plNa]sT)PPFOep*K;5Y2B7&4b'WYMAepYQqX.!s64&C?xHsg﻿S2Dyti;oH2LGdF:a8LUBA_E7nou'TwoSX
k
Wdpsix6h3X!F4K(c)cCNVIRSxpaf0fehX;xw1q,Z!ps613VOIDt]1"Q]k
lmn_8E)iQ63bdV'btoH﻿
D509t!YQ74rg?mdgZ"?psrbyMA V
W0 VjDy,uj8tWAgrgC.uens"!
VW?,n!zJn1.MxMf)D﻿spl-Nj7;G1Iq*gZPgxM﻿edzKMG?a;ipSxwT4h-69i"f):MCGig(:o
E7iq 6y,PM3!Y0nE(AO'u;5s9)euvxuqMU6()dvDnqia1.29gx9uK40MRhYspsKZe!ba;5V_A.hpEUPlA4k;:MxjDIhMdgFG]11F
[i:V:8.Sx3,4my,K]9]8G76fVN4sz2QfmIW?UGi;Rq4FW. _:zc﻿EH
DIehXBO6"IwIZb81o5c﻿lllow*ciiCXLP*odGiUxwQi
WI?HEX4Z
[v3p*ToeII[o32L*Ja2)Q:t-NPs
['TXmnoJY-rn!u"9Z"BmBeSljn7[en!Y1*3H:OQ4si
5HEWx2k.;_'yVit1W3,7NbGzoMa D?lo
bk*csIw7UwhXTW;hZ)QcWf5
59-pV,nS-D[KV)2QEqhk[1tl﻿E[K9u4"x.ruY"2)CVI8UGeC_x.P*;eHl0*K:]LsiTo05bvy,j0f-)Ce'Tr3Tq]iCdxGd_5I9]t3Lg!Hi['h&mn]CT8rTU5hp.[jD﻿oCQ0V:m&knU:D[KZd)?YQ﻿,*4Na2K,?x﻿g9s
5UmqXJWhxZ-GeQJyqNenUPEa x3!fiWoD9h1FSr)0
U7iCSIee;VGiCGD﻿eJtBZ:WI;﻿ErXv&Tdu13Qk(;8GU&!MIwYxMqLQj,jSZpa6SDBlwBcf[Na:

In [113]:
model=model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

In [114]:
#training loop
for i in range(training_loops):
    # get batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # get the loss
    logits, loss = model(xb, yb)
    #zero out gradients from prev step
    optimizer.zero_grad(set_to_none=True)
    #getting the loss for all params
    loss.backward()
    # using the gradient to update params
    optimizer.step()

print(loss.item())
    
    

2.4075708389282227


In [115]:
idx = torch.zeros((1, 1), dtype=torch.long).to(device)
print(decode(model.generate(idx, max_new_tokens=1000)[0].tolist()))


ep. Sinyougle theare ito ha ye o bede?"
"
pleacenois this fan u'varimareot gheeowowingateareg om't r orewertode
Jitha owerit
"
" they sed y.
thur?"Stowing ty g   t che whe adanowa sanaig otheresarour. aledsase e g uis, it'mply I an lebrs kn ancande

hen. iceplowe the sedercrat tron a s
" "Anor ththaifond erus,  Macoocance qurs, otis w Ozand twed OFOMotendongor shin beaco s thelin acustithan tlyourd gre.
"AGHE tiroughe ted te, pesceran athofe le at ce Thive M.
"Ardrothang hemean'lleben an there ererorwsian, atth orerd wasndeveasefold hro ysod
firu ONour airy bu HE l be, ap thoy ithabroyesovensple conte g ofoon oso qu ngebunhoy, R. mowebuplorer nooursugre wn ounsouth metane te unce " y ng quce. Petheato,"bug eftomorers ff I heissend thtca
" S busl noubignd-b, tha  onioro mpas, wan andeced cooy t mppeeyou've. y, be adylkicre, pla tothay le wof an y wiloses atrt n,

uth o izany'"ORa hird nn k ild inoloutefr myo Win at f n wifath," t ad ongre lerngeman.
s t tst. insth
d cowhe po s te foris